In [ ]:
# ======================================================
# Notebook: 5D Recipe Optimisation (SVM surrogate)
# Inputs: (20,5) | Output: (20,)
# Goal: maximise (- total negative score)
# ======================================================

import numpy as np
from sklearn.svm import SVR

# Load data
X = np.load("/mnt/data/initial_inputs.npy")      # (20,5)
y_raw = np.load("/mnt/data/initial_outputs.npy") # (20,)

# Transform objective
y = -y_raw

# Train SVM regression surrogate
model = SVR(kernel="rbf", C=10.0, gamma="scale")
model.fit(X, y)

# Candidate sampling
bounds = [(X[:,i].min(), X[:,i].max()) for i in range(5)]
n_candidates = 5000
X_grid = np.column_stack([
    np.random.uniform(b[0], b[1], n_candidates) for b in bounds
])

# Predict score
preds = model.predict(X_grid)

# Exploration term
dist = np.min(np.linalg.norm(X_grid[:,None,:] - X[None,:,:], axis=2), axis=1)

# Acquisition
acquisition = preds + 0.1 * dist

# Select next (10,5)
top_idx = np.argsort(acquisition)[-10:]
next_points = X_grid[top_idx]

print("Next (10,5) candidate recipes:")
print(next_points)